* for soybean may 1st to oct 30, experiment with different subsets (march - august)
* radar vegetation index
* VH/VV ratio
* will get the ablated dataset
* python's xgboost library
* 0.58 R^2
* target 0.6 R^2
* plot predicted vs actual
* look at error at the perimeter and get rid of them
* maybe use classification and see what results I get

# SAR feature data set only 

In [6]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/wheat_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "BASE" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 495789 rows with 215 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 160 features...
🚀 Mode: BASE | Features: 162 | Device: H200 GPU
Fold 1 | R2: 0.566 | RMSE: 15.86 | MAPE: 17.53% | Time: 1.5s
Fold 2 | R2: 0.453 | RMSE: 15.29 | MAPE: 17.65% | Time: 1.4s
Fold 3 | R2: 0.552 | RMSE: 15.98 | MAPE: 22.91% | Time: 1.3s
Fold 4 | R2: 0.557 | RMSE: 14.01 | MAPE: 17.39% | Time: 2.9s
Fold 5 | R2: 0.199 | RMSE: 20.99 | MAPE: 40.12% | Time: 0.8s

📊 FINAL SUMMARY: BASE
Total Time: 8.57s
-----------------------------------
R²:    0.4655 ± 0.1557
MAE:   12.49
RMSE:  16.43
NRMSE: 9.97%
MAPE:  23.12%
Acc:   0.4829
-----------------------------------


# SAR feature data set with weather 

In [7]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/wheat_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "WEATHER" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 495789 rows with 215 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 160 features...
🚀 Mode: WEATHER | Features: 183 | Device: H200 GPU
Fold 1 | R2: 0.536 | RMSE: 16.40 | MAPE: 18.75% | Time: 1.2s
Fold 2 | R2: 0.455 | RMSE: 15.27 | MAPE: 16.22% | Time: 2.0s
Fold 3 | R2: 0.515 | RMSE: 16.62 | MAPE: 23.12% | Time: 1.4s
Fold 4 | R2: 0.520 | RMSE: 14.59 | MAPE: 17.98% | Time: 2.7s
Fold 5 | R2: 0.322 | RMSE: 19.32 | MAPE: 35.17% | Time: 2.7s

📊 FINAL SUMMARY: WEATHER
Total Time: 10.61s
-----------------------------------
R²:    0.4696 ± 0.0882
MAE:   12.60
RMSE:  16.44
NRMSE: 9.98%
MAPE:  22.25%
Acc:   0.4888
-----------------------------------


# SAR feature data set with weather and terrain features 

In [8]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/wheat_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "TERRAIN" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 495789 rows with 215 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 160 features...
🚀 Mode: TERRAIN | Features: 192 | Device: H200 GPU
Fold 1 | R2: 0.524 | RMSE: 16.61 | MAPE: 19.13% | Time: 1.1s
Fold 2 | R2: 0.461 | RMSE: 15.17 | MAPE: 16.41% | Time: 2.3s
Fold 3 | R2: 0.529 | RMSE: 16.38 | MAPE: 22.66% | Time: 1.5s
Fold 4 | R2: 0.503 | RMSE: 14.84 | MAPE: 17.98% | Time: 3.4s
Fold 5 | R2: 0.236 | RMSE: 20.50 | MAPE: 39.72% | Time: 1.2s

📊 FINAL SUMMARY: TERRAIN
Total Time: 10.07s
-----------------------------------
R²:    0.4508 ± 0.1229
MAE:   12.79
RMSE:  16.70
NRMSE: 10.14%
MAPE:  23.18%
Acc:   0.4704
-----------------------------------


# SAR features data set with weather, terrain features, and spatial feature

In [5]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/wheat_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 495789 rows with 215 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 160 features...
🚀 Mode: FULL_SPATIAL | Features: 513 | Device: H200 GPU
Fold 1 | R2: 0.557 | RMSE: 16.03 | MAPE: 18.07% | Time: 1.8s
Fold 2 | R2: 0.555 | RMSE: 13.79 | MAPE: 15.29% | Time: 2.7s
Fold 3 | R2: 0.634 | RMSE: 14.45 | MAPE: 19.16% | Time: 4.1s
Fold 4 | R2: 0.597 | RMSE: 13.37 | MAPE: 16.03% | Time: 3.4s
Fold 5 | R2: 0.300 | RMSE: 19.63 | MAPE: 30.89% | Time: 2.5s

📊 FINAL SUMMARY: FULL_SPATIAL
Total Time: 15.03s
-----------------------------------
R²:    0.5284 ± 0.1319
MAE:   11.93
RMSE:  15.46
NRMSE: 9.38%
MAPE:  19.89%
Acc:   0.5112
-----------------------------------


# SAR features data set with weather, terrain features, spatial features, and ablations 

In [2]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/wheat_2016_2023_processed.parquet"
ABLATED_PATH = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig['is_ablation'] = False

valid_years = df_orig['year'].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# FILTER: Soy only + Matching Years only
df_ablated = df_ablated[
    (df_ablated['crop'].str.contains('wheat', case=False, na=False)) & 
    (df_ablated['year'].isin(valid_years))
].copy()
df_ablated['is_ablation'] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated['farm_name'].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated['farm_name'] = df_ablated['farm_name'].map(farm_id_map)

# Combine datasets
common_cols = list(set(df_orig.columns) & set(df_ablated.columns))
df = pd.concat([df_orig[common_cols + ['is_ablation']], 
                df_ablated[common_cols + ['is_ablation']]], axis=0).reset_index(drop=True)

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(['farm_name', 'year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    for farm in df['farm_name'].unique():
        mask = df['farm_name'] == farm
        f_df = df[mask]
        if len(f_df) <= n: continue
        tree = BallTree(f_df[['latitude', 'longitude']].values)
        _, idx = tree.query(f_df[['latitude', 'longitude']].values, k=n+1)
        for i, col in enumerate(sar_cols):
            neighbor_data[mask, i] = f_df[col].values[idx[:, 1:]].mean(axis=1)
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    
    if len(real_te_idx) == 0: continue
    
    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]
    
    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = y[tr].mean()
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )
    
    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)
    
    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    
    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

Loaded 495789 real rows and 800201 filtered ablated rows.
Calculating Spatial Features...
Generating BallTree neighbor features for 80 features...
🚀 Mode: FULL_SPATIAL_WITH_ABLATION | Features: 272 | Device: GPU
Fold 1 | R2: 0.768 | RMSE: 10.41 | MAPE: 11.07% | Time: 8.1s
Fold 2 | R2: 0.737 | RMSE: 11.28 | MAPE: 13.58% | Time: 8.4s
Fold 3 | R2: 0.614 | RMSE: 14.96 | MAPE: 36.09% | Time: 8.3s
Fold 4 | R2: 0.767 | RMSE: 12.21 | MAPE: 12.73% | Time: 6.5s
Fold 5 | R2: 0.720 | RMSE: 10.70 | MAPE: 11.04% | Time: 8.4s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATION
Total Time: 41.21s
-----------------------------------
R²:    0.7213 ± 0.0631
MAE:   8.74
RMSE:  11.91
NRMSE: 7.15%
MAPE:  16.90%
Acc:   0.6588
-----------------------------------


# SAR features data set with weather, terrain features, spatial features, and ALL ablations

In [3]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/wheat_2016_2023_processed.parquet"

# NEW ablated file (SAR + TERRAIN + WEATHER ablated)
ABLATED_PATH = "data/gamma_k8_stacked_ablated_TERRAIN_WEATHER_corn_wheat_soy.parquet"

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False

valid_years = df_orig["year"].unique()

df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# FILTER: Soy only + Matching Years only
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains("wheat", case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_id_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# ---- IMPORTANT CHANGE: use UNION of columns, not intersection ----
# Keep yield in both, and align schemas by reindexing.
all_cols = sorted(set(df_orig.columns) | set(df_ablated.columns))
df_orig_aligned = df_orig.reindex(columns=all_cols)
df_ablated_aligned = df_ablated.reindex(columns=all_cols)

df = pd.concat([df_orig_aligned, df_ablated_aligned], axis=0, ignore_index=True)
print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
# SAR (as before)
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
SAR_ALL = SAR_RAW + SAR_INDICES

# ---- IMPORTANT CHANGE: include *_season weather too ----
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"])]
# (this will include prcp_04..prcp_10 AND prcp_season, etc.)

TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL = [c for c in df.columns if c.startswith("soil_")]

# Only compute spatial features from SAR (keeps your design intact)
print("Calculating Spatial Features...")
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors_fast(df, cols, n=8, only_real=True):
    """
    Vectorized BallTree neighbor mean features.
    - Computes per-farm neighbor means for ALL columns at once.
    - Optionally computes ONLY for real rows (recommended).
    """
    print(f"Generating BallTree neighbor features for {len(cols)} features... (n={n}, only_real={only_real})")

    # Output array (float32, NaN default)
    out = np.full((len(df), len(cols)), np.nan, dtype=np.float32)

    # We'll optionally skip ablated rows to save huge time
    if only_real and "is_ablation" in df.columns:
        base_mask = ~df["is_ablation"].astype(bool)
    else:
        base_mask = pd.Series(True, index=df.index)

    # Iterate farms (still needed for separate trees), but vectorized inside
    for farm, idx in df.loc[base_mask].groupby("farm_name").groups.items():
        idx = np.array(list(idx), dtype=int)
        if idx.size <= n:
            continue

        f_df = df.loc[idx]
        coords = f_df[["latitude", "longitude"]].to_numpy(dtype=np.float64)

        # Build tree and query neighbors
        tree = BallTree(coords)
        _, nn = tree.query(coords, k=n+1)       # nn shape: (m, n+1), includes self
        nn = nn[:, 1:]                          # drop self -> (m, n)

        # Feature matrix (m, p)
        A = f_df[cols].to_numpy(dtype=np.float32)

        # Gather neighbors: (m, n, p)
        neigh = A[nn]  # uses fancy indexing

        # NaN-safe mean along neighbor axis
        s = np.nansum(neigh, axis=1)  # (m, p)
        c = np.sum(~np.isnan(neigh), axis=1)  # (m, p)
        m = s / np.maximum(c, 1)  # avoid division by 0
        m[c == 0] = np.nan        # where all neighbors are NaN -> keep NaN (no warning)

        out[idx, :] = m

    new_cols = [f"{c}_neighbor_{n}" for c in cols]
    df = pd.concat([df, pd.DataFrame(out, columns=new_cols, index=df.index)], axis=1)
    return df, new_cols


df, SAR_NEIGHBORS = add_spatial_neighbors_fast(df, SAR_ALL, n=8, only_real=True)


# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER"

SELECTED_FEATURES = (
    SAR_ALL
    + list(field_means.columns)
    + SAR_NEIGHBORS
    + WEATHER
    + TERRAIN
    + SOIL
    + ["year", "latitude"]
)

# Guard against any missing cols (just in case)
SELECTED_FEATURES = [c for c in SELECTED_FEATURES if c in df.columns]

X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False, duplicates="drop")
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()

    # Test only on REAL fields
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    if len(real_te_idx) == 0:
        continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]

    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = y[tr].mean()
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        early_stopping_rounds=50,
    )

    model.fit(
        X_tr,
        y[tr] / mu_tr,
        eval_set=[(X_te_real, y_te_real / mu_te_real)],
        verbose=False
    )

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0

    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({"R2": r2, "MAE": mae, "RMSE": rmse, "NRMSE": nrmse, "MAPE": mape, "Zone_Acc": acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 495789 real rows and 800201 filtered ablated rows.
Calculating Spatial Features...
Generating BallTree neighbor features for 640 features... (n=8, only_real=True)
🚀 Mode: FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER | Features: 1955 | Device: GPU
Fold 1 | R2: 0.781 | RMSE: 10.99 | MAPE: 12.26% | Time: 189.7s
Fold 2 | R2: 0.797 | RMSE: 11.85 | MAPE: 17.06% | Time: 29.9s
Fold 3 | R2: 0.600 | RMSE: 14.18 | MAPE: 30.34% | Time: 28.8s
Fold 4 | R2: 0.741 | RMSE: 11.95 | MAPE: 12.34% | Time: 36.8s
Fold 5 | R2: 0.581 | RMSE: 12.46 | MAPE: 12.50% | Time: 33.6s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER
Total Time: 320.36s
-----------------------------------
R²:    0.6996 ± 0.1023
MAE:   9.09
RMSE:  12.29
NRMSE: 7.48%
MAPE:  16.90%
Acc:   0.6493
-----------------------------------


# Making the prediction column

In [1]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/wheat_2016_2023_processed.parquet"
ABLATED_PATH  = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

OUTPUT_PATH   = "data/wheat_full_spatial_with_ablation__with_oof_preds.parquet"
PRED_COL      = "pred_yield_oof"
PRED_FOLD_COL = "pred_fold"

TARGET_CROP = "wheat"  # ✅ IMPORTANT

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False

valid_years = df_orig["year"].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# ✅ FILTER: CORN only + Matching Years only
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains(TARGET_CROP, case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# Combine datasets (deterministic column order + no duplicate is_ablation)
common_cols = [c for c in df_orig.columns if c in df_ablated.columns]
# Ensure is_ablation appears exactly once and at the end (nice for readability)
common_cols = [c for c in common_cols if c != "is_ablation"] + ["is_ablation"]

df = pd.concat(
    [df_orig[common_cols], df_ablated[common_cols]],
    axis=0
).reset_index(drop=True)

# Safety: drop any accidental duplicates (should be none now)
df = df.loc[:, ~df.columns.duplicated()]

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows for crop='{TARGET_CROP}'.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df_in, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df_in), len(sar_cols)), dtype=np.float32)

    for farm in df_in["farm_name"].unique():
        mask = (df_in["farm_name"] == farm).to_numpy()
        f_df = df_in.loc[mask, :]
        if len(f_df) <= n:
            continue

        coords = f_df[["latitude", "longitude"]].to_numpy()
        tree = BallTree(coords)
        _, idx = tree.query(coords, k=n + 1)  # includes self at idx[:,0]

        for i, col in enumerate(sar_cols):
            vals = f_df[col].to_numpy()
            neighbor_data[mask, i] = vals[idx[:, 1:]].mean(axis=1)

    new_cols = [f"{c}_neighbor_{n}" for c in sar_cols]
    df_out = pd.concat([df_in, pd.DataFrame(neighbor_data, columns=new_cols, index=df_in.index)], axis=1)
    return df_out, new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL, n=8)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]

X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# Allocate prediction columns
df[PRED_COL] = np.nan
df[PRED_FOLD_COL] = np.nan

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()

    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    if len(real_te_idx) == 0:
        continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]

    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = float(y[tr].mean())
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )

    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Store OOF predictions
    df.loc[real_te_idx, PRED_COL] = p_raw.astype(np.float32)
    df.loc[real_te_idx, PRED_FOLD_COL] = k

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0

    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

# ---------------- 6. SAVE DATASET WITH PREDICTIONS ----------------
df = df.loc[:, ~df.columns.duplicated()]
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n✅ Saved dataframe (with OOF predictions) to: {OUTPUT_PATH}")
print(f"Columns added: {PRED_COL}, {PRED_FOLD_COL}")

Loaded 495789 real rows and 800201 filtered ablated rows for crop='wheat'.
Calculating Spatial Features...
Generating BallTree neighbor features for 80 features...
🚀 Mode: FULL_SPATIAL_WITH_ABLATION | Features: 272 | Device: GPU
Fold 1 | R2: 0.835 | RMSE: 9.53 | MAPE: 10.99% | Time: 9.2s
Fold 2 | R2: 0.832 | RMSE: 10.77 | MAPE: 15.04% | Time: 8.6s
Fold 3 | R2: 0.661 | RMSE: 13.04 | MAPE: 28.30% | Time: 8.6s
Fold 4 | R2: 0.777 | RMSE: 11.09 | MAPE: 11.31% | Time: 8.7s
Fold 5 | R2: 0.679 | RMSE: 10.90 | MAPE: 10.21% | Time: 8.3s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATION
Total Time: 45.06s
-----------------------------------
R²:    0.7567 ± 0.0829
MAE:   7.97
RMSE:  11.07
NRMSE: 6.74%
MAPE:  15.17%
Acc:   0.7009
-----------------------------------

✅ Saved dataframe (with OOF predictions) to: data/wheat_full_spatial_with_ablation__with_oof_preds.parquet
Columns added: pred_yield_oof, pred_fold
